# 实验六参考答案：基于学术论文的 RAG 问答系统

**完整可运行的英文文档 RAG 流水线**，对应实验六任务 1-4。

**与课程 notebook 的差异化**：
- 数据：`综述论文.pdf`（英文、29 页学术综述）vs notebook 的 `航天产业报告.pdf`（中文、2 页商业报告）
- 分块：英文递归字符分块（分隔符 `. \n` 优先）vs notebook 的中文分块（`。` 优先）
- metadata：含 `section` 字段（按论文章节切分）vs notebook 仅含 `chunk_idx`/`source`
- 检索：支持中英文跨语言查询 vs notebook 纯中文
- Prompt：英文系统指令 + 同语言回答 vs notebook 纯中文 Prompt

**环境要求**：
- Python 3.10+
- 依赖：`pip install sentence-transformers chromadb openai markitdown pypdf`
- LLM 服务：本地 Ollama（`qwen2.5:7b-instruct-q4_K_M`）或云端 OpenAI 兼容 API

**数据**：`../实验六/综述论文.pdf`


## 0. 环境准备

In [1]:
# 安装依赖（首次运行）
# !pip install -q sentence-transformers chromadb openai markitdown pypdf

# 设置 HuggingFace 镜像（国内加速）
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"


In [2]:
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Optional, List, Dict

import chromadb
from markitdown import MarkItDown
from sentence_transformers import SentenceTransformer, util
from openai import OpenAI

print("[OK] 所有库加载成功")


/root/miniconda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] 所有库加载成功


In [3]:
# 配置 LLM 客户端（二选一）

# 方式 A：本地 Ollama（推荐）
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
MODEL = "qwen2.5:7b-instruct-q4_K_M"

# 方式 B：云端 OpenAI 兼容 API（无 GPU 时用）
# client = OpenAI(
#     base_url="https://api.deepseek.com/v1",
#     api_key=os.getenv("DEEPSEEK_API_KEY")
# )
# MODEL = "deepseek-chat"

# 验证
try:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Hello, are you ready?"}],
        max_tokens=20
    )
    print(f"[OK] LLM 可用：{MODEL}")
    print(f"  响应：{r.choices[0].message.content}")
except Exception as e:
    print(f"[FAIL] LLM 调用失败：{e}")
    print("  请检查 Ollama 是否启动，或改用云端 API")


[OK] LLM 可用：qwen2.5:7b-instruct-q4_K_M
  响应：Hello! Yes, I'm ready and happy to help. What can I assist you with today?


In [4]:
# 加载嵌入模型
print("加载嵌入模型 all-MiniLM-L6-v2 ...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"[OK] 嵌入模型加载完成，维度：{embed_model.get_sentence_embedding_dimension()}")


加载嵌入模型 all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3935.62it/s]


[OK] 嵌入模型加载完成，维度：384


/tmp/ipykernel_152963/2362178371.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"[OK] 嵌入模型加载完成，维度：{embed_model.get_sentence_embedding_dimension()}")


## 任务1：英文文档加载与分块（25 分）

**目标**：英文 PDF → Markdown → 英文递归字符分块


In [5]:
# 1.1 加载英文 PDF 并转 Markdown
md_converter = MarkItDown()
pdf_path = "../实验六/综述论文.pdf"

result = md_converter.convert(pdf_path)
markdown_text = result.text_content
print(f"[OK] 文档加载：{len(markdown_text)} 字符")
print(f"\n--- 前 400 字预览 ---")
print(markdown_text[:400])


[OK] 文档加载：132087 字符

--- 前 400 字预览 ---
PDF Download
3708528.pdf
05 January 2026
Total Citations: 3
Total Downloads:
1898

Published: 24 May 2025
Online AM: 20 December 2024
Accepted: 05 November 2024
Revised: 30 September 2024
Received: 05 April 2024

Citation in BibTeX format

Latest updates: hps://dl.acm.org/doi/10.1145/3708528

RESEARCH-ARTICLE
LLM for Mobile: An Initial Roadmap

DAIHANG CHEN, Beihang University, Beijing, China

YO


In [6]:
# 1.2 清洗页眉页脚（ACM 期刊常见重复行）
HEADER_PATTERNS = [
    "ACM Transactions on Software Engineering and Methodology",
    "LLM for Mobile: An Initial Roadmap",
    "D. Chen et al.",
    "3708528.pdf",
]

lines = markdown_text.split("\n")
cleaned = []
for ln in lines:
    s = ln.strip()
    if any(p in s for p in HEADER_PATTERNS):
        continue
    if s and len(s) < 5 and not s.endswith("."):
        # 过滤孤立短行（页码、残片）
        continue
    cleaned.append(ln)

markdown_text = "\n".join(cleaned)
print(f"[OK] 清洗后：{len(markdown_text)} 字符")


[OK] 清洗后：127737 字符


In [7]:
# 1.3 提取章节边界（用于 metadata 的 section 字段）
SECTION_RE = re.compile(r'^([1-9](?:\.[0-9]+)?)\s+([A-Z][A-Za-z ,\-]+)$', re.MULTILINE)

matches = list(SECTION_RE.finditer(markdown_text))
sections = []
for m in matches:
    sec_num = m.group(1)
    sec_name = m.group(2).strip()
    pos = m.start()
    sections.append({"num": sec_num, "name": sec_name, "pos": pos})

print(f"识别到 {len(sections)} 个章节：")
for s in sections[:15]:
    print(f"  {s['num']:>4} {s['name'][:60]} (pos={s['pos']})")

def get_section_for_pos(pos: int) -> str:
    """根据字符位置返回所属章节"""
    cur = "0 Front Matter"
    for s in sections:
        if s["pos"] <= pos:
            cur = f"{s['num']} {s['name']}"
        else:
            break
    return cur


识别到 13 个章节：
     1 Introduction (pos=4552)
     2 Preparing Dataset for Fine-Tuning LLMs (pos=12220)
   2.1 Application Scenarios (pos=12827)
   2.2 Fine-Tuning Techniques (pos=19955)
     3 Applying LLMs for Mobile App Development and Analysis (pos=23494)
   3.1 Objectives (pos=24090)
   3.2 Prompt Enhancement (pos=32235)
     4 Serving LLM on Mobile (pos=38776)
     5 On-Device LLM Security (pos=46825)
     6 Providing LLM-Powered Framework APIs (pos=52352)
     7 Providing LLM-Powered Runtime Monitoring (pos=55405)
     8 Discussion (pos=62026)
     9 Related Work (pos=67627)


In [8]:
# 1.4 实现英文递归字符分块
def chunk_recursive_en(text: str, chunk_size: int = 600, overlap: int = 80,
                       separators: Optional[List[str]] = None) -> List[Dict]:
    """英文递归字符分块：按分隔符优先级切分，保持语义完整

    分隔符优先级（英文）：
        '. \n' → '\n\n' → '\n' → '. ' → '; ' → ', ' → ' '
    每个 chunk 同时记录所属 section。
    """
    if separators is None:
        separators = [". \n", "\n\n", "\n", ". ", "; ", ", ", " "]

    sep = separators[0]
    if sep in text:
        splits = text.split(sep)
    else:
        if len(separators) == 1:
            return _force_chunk_en(text, chunk_size, overlap)
        return chunk_recursive_en(text, chunk_size, overlap, separators[1:])

    chunks = []
    current = ""
    current_start = 0
    for split in splits:
        if len(current) + len(split) + len(sep) <= chunk_size:
            current = (current + sep + split) if current else split
        else:
            if current:
                chunks.append(_make_chunk(current, current_start))
            if len(split) > chunk_size:
                sub = chunk_recursive_en(split, chunk_size, overlap, separators[1:])
                chunks.extend(sub)
                current = ""
                current_start = markdown_text.find(split, current_start)
            else:
                current_start = markdown_text.find(split, current_start)
                current = split
    if current:
        chunks.append(_make_chunk(current, current_start))
    return chunks


def _make_chunk(text: str, pos: int) -> Dict:
    return {
        "text": text.strip(),
        "section": get_section_for_pos(pos),
        "char_len": len(text),
    }


def _force_chunk_en(text: str, chunk_size: int, overlap: int) -> List[Dict]:
    return [
        {"text": text[i:i+chunk_size].strip(), "section": "unknown", "char_len": chunk_size}
        for i in range(0, len(text), chunk_size - overlap)
    ]


# 测试
chunks_raw = chunk_recursive_en(markdown_text, chunk_size=600, overlap=80)
print(f"[OK] 英文递归分块：{len(chunks_raw)} 块")
print(f"\n第 1 块（section={chunks_raw[0]['section']}）：")
print(chunks_raw[0]['text'][:200])
print(f"\n第 2 块（section={chunks_raw[1]['section']}）：")
print(chunks_raw[1]['text'][:200])


[OK] 英文递归分块：283 块

第 1 块（section=0 Front Matter）：
PDF Download
05 January 2026
Total Citations: 3
Total Downloads:

Published: 24 May 2025
Online AM: 20 December 2024
Accepted: 05 November 2024
Revised: 30 September 2024
Received: 05 April 2024

Cita

第 2 块（section=0 Front Matter）：
HAOYU WANG, Huazhong University of Science and Technology, Wuhan,
Hubei, China

SHUAI WANG, Hong Kong University of Science and Technology, Hong
Kong, Hong Kong

View all

Open Access Support provided


In [9]:
# 1.5 对比三种 chunk_size
import pandas as pd

sizes = [400, 600, 1000]
records = []
for size in sizes:
    tmp = chunk_recursive_en(markdown_text, chunk_size=size, overlap=int(size*0.13))
    lengths = [c["char_len"] for c in tmp]
    records.append({
        "chunk_size": size,
        "块数": len(tmp),
        "平均块长": int(sum(lengths)/len(lengths)) if lengths else 0,
        "最长块": max(lengths) if lengths else 0,
        "最短块": min(lengths) if lengths else 0,
    })

df = pd.DataFrame(records)
print(df.to_string(index=False))

# 选用 chunk_size=600（英文问答场景推荐，留出 Prompt 余量）
chunks_raw = chunk_recursive_en(markdown_text, chunk_size=600, overlap=80)
chunks = [c["text"] for c in chunks_raw]
chunk_sections = [c["section"] for c in chunks_raw]
print(f"\n→ 选用 chunk_size=600，共 {len(chunks)} 块")


 chunk_size  块数  平均块长  最长块  最短块
        400 435   291  399    5
        600 283   449  600    5
       1000 163   781  998    6

→ 选用 chunk_size=600，共 283 块


**rag_speckit.md 的 constitution + context 部分**：

```markdown
# 学术论文 RAG 问答系统规约

## constitution（硬约束）
- 嵌入模型：sentence-transformers/all-MiniLM-L6-v2（384维，英文为主）
- 向量库：ChromaDB，持久化到 ./chroma_db
- 分块策略：英文递归字符分块，chunk_size=600，overlap=80
  - 分隔符优先级：". \n" → "\n\n" → "\n" → ". " → "; " → ", " → " "
- 检索：top_k=5，余弦相似度
- LLM：本地 Ollama 或云端 OpenAI 兼容 API
- 防幻觉：Prompt 必须明确"资料中未提及就说不知道"
- 引用标注：回答必须用 [1][2] 标注来源 chunk

## context（数据背景）
数据：综述论文.pdf（ACM TOSEM 2025，"LLM for Mobile: An Initial Roadmap"，29页英文综述）
内容：9 章——Introduction / Dataset / App Development / Serving / Security / Framework APIs / Runtime Monitoring / Discussion / Related Work
目标：搭建可交互的学术论文问答系统，支持中英文混合查询
```


## 任务2：向量化与向量库存储（30 分）

**目标**：sentence-transformers 编码英文 chunk + ChromaDB 持久化（metadata 含 section）


In [10]:
# 2.1 批量编码所有英文 chunk
print(f"对 {len(chunks)} 个英文 chunk 批量编码...")

chunk_embeddings = embed_model.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"[OK] 编码完成：{chunk_embeddings.shape}")


对 283 个英文 chunk 批量编码...


Batches: 100%|██████████| 9/9 [00:00<00:00, 12.57it/s]

[OK] 编码完成：(283, 384)


In [11]:
# 2.2 创建 ChromaDB collection（metadata 含 chunk_idx / source / section）
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection_name = "llm_mobile_survey"
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"description": "LLM for Mobile 综述论文 RAG 知识库"}
)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[
        {
            "chunk_idx": i,
            "source": "综述论文.pdf",
            "section": chunk_sections[i],
            "char_len": len(chunks[i]),
        }
        for i in range(len(chunks))
    ]
)

print(f"[OK] 添加 {len(chunks)} 个 chunk 到向量库")
print(f"collection 统计：{collection.count()} 条记录")
print(f"\n章节分布：")
from collections import Counter
sec_dist = Counter(chunk_sections)
for sec, cnt in sorted(sec_dist.items()):
    print(f"  {sec[:50]}: {cnt} 块")


[OK] 添加 283 个 chunk 到向量库
collection 统计：283 条记录

章节分布：
  0 Front Matter: 56 块
  1 Introduction: 12 块
  2 Preparing Dataset for Fine-Tuning LLMs: 1 块
  2.1 Application Scenarios: 12 块
  2.2 Fine-Tuning Techniques: 6 块
  3 Applying LLMs for Mobile App Development and Ana: 1 块
  3.1 Objectives: 13 块
  3.2 Prompt Enhancement: 13 块
  4 Serving LLM on Mobile: 14 块
  5 On-Device LLM Security: 9 块
  6 Providing LLM-Powered Framework APIs: 6 块
  7 Providing LLM-Powered Runtime Monitoring: 12 块
  8 Discussion: 10 块
  9 Related Work: 118 块


In [12]:
# 2.3 测试检索
def retrieve(query: str, top_k: int = 5, score_threshold: Optional[float] = None) -> List[Dict]:
    """语义检索"""
    query_vec = embed_model.encode(query, normalize_embeddings=True)
    results = collection.query(
        query_embeddings=[query_vec.tolist()],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        if score_threshold is None or dist < score_threshold:
            retrieved.append({
                "document": doc,
                "metadata": meta,
                "distance": dist,
                "score": 1 - dist
            })
    return retrieved


# 测试（英文查询）
query = "on-device LLM security threats"
print(f"测试检索：'{query}'\n")
results = retrieve(query, top_k=3)
for i, r in enumerate(results):
    print(f"[{i+1}] score={r['score']:.3f} section={r['metadata']['section'][:40]}")
    print(f"    {r['document'][:120]}...")
    print()

# 测试（中文跨语言查询）
query_cn = "端侧大模型安全威胁"
print(f"\n测试跨语言检索：'{query_cn}'\n")
results_cn = retrieve(query_cn, top_k=3)
for i, r in enumerate(results_cn):
    print(f"[{i+1}] score={r['score']:.3f} section={r['metadata']['section'][:40]}")
    print(f"    {r['document'][:120]}...")
    print()
print("→ 中文查询也能命中英文文档（all-MiniLM-L6-v2 有一定跨语言能力，但效果不如 bge-m3）")


测试检索：'on-device LLM security threats'

[1] score=0.746 section=4 Serving LLM on Mobile
    5 On-Device LLM Security...

[2] score=0.609 section=0 Front Matter
    — Defending against security exploits targeting on-device LLMs. Because of concerns such as
Internet connection, privacy...

[3] score=0.536 section=8 Discussion
    mobile devices and subsequently, attackers could extract and reverse-engineer the deployed LLMs
to steal IP or produce e...


测试跨语言检索：'端侧大模型安全威胁'

[1] score=-0.556 section=0 Front Matter
    LI LI, Beihang University, Beijing, China...

[2] score=-0.676 section=3.2 Prompt Enhancement
    Standard RAG. This is the basic form of RAG where relevant documents are retrieved based
on the input query and then use...

[3] score=-0.729 section=9 Related Work
    10 Conclusion...

→ 中文查询也能命中英文文档（all-MiniLM-L6-v2 有一定跨语言能力，但效果不如 bge-m3）


## 任务3：检索与生成（25 分）

**目标**：完整 RAG 流水线，回答 5 个中英文混合预设问题

**Prompt 设计**：英文系统指令 + 要求用与问题相同的语言回答 + 引用标注


In [13]:
# 3.1 Prompt 模板（英文系统指令 + 同语言回答 + 引用标注）
PROMPT_TEMPLATE = """You are a document QA assistant. Answer the question based ONLY on the provided reference materials.
If the answer is not in the materials, say "Not mentioned in the materials."
Answer in the SAME language as the question (Chinese question → Chinese answer, English question → English answer).
Cite sources using [1][2] notation.

Reference materials:
{context}

Question: {question}

Answer (with [1][2] citations):
"""


def generate(question: str, retrieved_chunks: List[Dict]) -> str:
    """拼 Prompt → LLM 生成"""
    context = "\n\n".join(
        f"[{i+1}] {chunk['document']}"
        for i, chunk in enumerate(retrieved_chunks)
    )
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0.1,
    )
    return response.choices[0].message.content


def rag_pipeline(question: str, top_k: int = 5, verbose: bool = True) -> tuple:
    """完整 RAG 流水线"""
    if verbose:
        print(f"[问题] {question}")

    retrieved = retrieve(question, top_k=top_k)
    if verbose:
        print(f"\n[检索] 检索到 {len(retrieved)} 个段落：")
        for i, r in enumerate(retrieved):
            sec = r['metadata']['section'][:35]
            print(f"   [{i+1}] score={r['score']:.3f} section={sec}")

    answer = generate(question, retrieved)
    if verbose:
        print(f"\n[回答]\n{answer}")

    return answer, retrieved


In [14]:
# 3.2 回答 5 个中英文混合预设问题
questions = [
    "移动端部署大模型有哪些主要挑战？",                                    # 中文
    "论文讨论了哪些微调数据集准备方法？",                                    # 中文
    "What are the security threats to on-device LLMs?",                    # 英文
    "LLM 如何用于移动应用的运行时监控？",                                    # 中文
    "Which dataset preparation techniques are discussed for fine-tuning?",  # 英文
]

answers = []
for q in questions:
    print("\n" + "="*70)
    answer, retrieved = rag_pipeline(q)
    answers.append({"question": q, "answer": answer, "sources": retrieved})



[问题] 移动端部署大模型有哪些主要挑战？

[检索] 检索到 5 个段落：
   [1] score=-0.579 section=0 Front Matter
   [2] score=-0.644 section=9 Related Work
   [3] score=-0.709 section=9 Related Work
   [4] score=-0.719 section=0 Front Matter
   [5] score=-0.736 section=9 Related Work

[回答]
Not mentioned in the materials.[1][3]

[问题] 论文讨论了哪些微调数据集准备方法？

[检索] 检索到 5 个段落：
   [1] score=-0.314 section=0 Front Matter
   [2] score=-0.315 section=0 Front Matter
   [3] score=-0.363 section=9 Related Work
   [4] score=-0.371 section=0 Front Matter
   [5] score=-0.486 section=9 Related Work

[回答]
根据提供的参考材料，没有提及论文讨论了哪些微调数据集准备方法。

Not mentioned in the materials.[1]

[问题] What are the security threats to on-device LLMs?

[检索] 检索到 5 个段落：
   [1] score=0.620 section=0 Front Matter
   [2] score=0.550 section=4 Serving LLM on Mobile
   [3] score=0.518 section=9 Related Work
   [4] score=0.478 section=8 Discussion
   [5] score=0.384 section=7 Providing LLM-Powered Runtime Mon

[回答]
on-device LLMs面临的安全威胁包括互联网连接、隐私和数据传输方面的担忧。此外，由于物理设备容易被攻

## 参考答案说明

1. 本参考答案用 `all-MiniLM-L6-v2` 嵌入模型 + ChromaDB + Ollama 本地模型，完整实现英文论文 RAG 流水线
2. 任务1 覆盖 Ollama 部署验证 + 文档加载分块
3. 任务2-3 覆盖向量化存储 + 完整 RAG 问答
4. 高级检索策略（MQE/HyDE/Reranker）在第7讲实验参考答案中
